# 01 Acquire

One section per modality. Each section requests the data, saves the raw response unchanged to its own folder in `data/raw/`, then checks what was saved.

# All imports

In [ ]:
import requests
import os
import json
import time
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv

---
# 1. API: TheirStack Jobs API

Load the key from the .env file to the notebook

In [13]:
load_dotenv("../../.env")

api_key = os.environ.get("THEIRSTACK_API_KEY")
if api_key:
    print("key loaded")
else:
    print("key not found, check the .env file")

key loaded


Build one small test request

In [14]:
url = "https://api.theirstack.com/v1/jobs/search"
headers = {"Authorization": f"Bearer {api_key}"}

body = {
    "posted_at_max_age_days": 90,
    "job_country_code_or": ["CA"],
    "job_title_or": ["data analyst"],
    "limit": 5,
}

print(url)
print(body)

https://api.theirstack.com/v1/jobs/search
{'posted_at_max_age_days': 90, 'job_country_code_or': ['CA'], 'job_title_or': ['data analyst'], 'limit': 5}


Send the request, check the status, save the raw response

In [15]:
test_file = "../data/raw/theirstack/theirstack_ca_data_analyst_test.json"

if os.path.exists(test_file):
    print("already saved, skipping:", test_file)
else:
    response = requests.post(url, headers=headers, json=body, timeout=30)
    print(response.status_code)

    if response.status_code == 200:
        data = response.json()
        with open(test_file, "w") as f:
            json.dump(data, f, indent=2)
        print(datetime.now(), "| TheirStack | data analyst test |", len(data["data"]), "postings saved to", test_file)
    else:
        print(response.text)

200
2026-11-22 19:44:36.828955 | TheirStack | data analyst test | 5 postings saved to ../data/raw/theirstack/theirstack_ca_data_analyst_test.json


Check whats inside one job posting

In [16]:
with open(test_file) as f:
    test = json.load(f)

print(test.keys())
print(test["metadata"])

first = test["data"][0]
print(sorted(first.keys()))
print()
print(first["job_title"], "|", first["company"], "|", first["location"], "|", first["date_posted"])
print(first["latitude"], first["longitude"])
print(first["description"][:500])

dict_keys(['metadata', 'data'])
{'total_results': None, 'truncated_results': 0, 'truncated_companies': 0, 'total_companies': None}
['avg_annual_salary_usd', 'cities', 'closed_at', 'company', 'company_domain', 'company_object', 'continents', 'countries', 'country', 'country_code', 'country_codes', 'date_posted', 'date_reposted', 'description', 'discovered_at', 'easy_apply', 'employment_statuses', 'final_url', 'has_blurred_data', 'hiring_team', 'hybrid', 'id', 'job_title', 'keyword_slugs', 'latitude', 'location', 'locations', 'long_location', 'longitude', 'manager_roles', 'matching_phrase_segments', 'matching_phrases', 'matching_words', 'max_annual_salary', 'max_annual_salary_usd', 'max_salary', 'min_annual_salary', 'min_annual_salary_usd', 'min_salary', 'normalized_title', 'postal_code', 'remote', 'reposted', 'salary_currency', 'salary_period', 'salary_string', 'seniority', 'short_location', 'source_url', 'state_code', 'technology_slugs', 'url']

Product Data Analyst, eCommerce [AQ-2782

Loop through Data Analyst, Engineer, and Scientist roles and save each one

In [18]:
roles = ["data analyst", "data engineer", "data scientist"]
pages = [0, 1]

for role in roles:
    for page in pages:
        filename = f"../data/raw/theirstack/theirstack_ca_{role.replace(' ', '_')}_p{page}.json"

        if os.path.exists(filename):
            print("already saved, skipping:", filename)
            continue

        body = {
            "posted_at_max_age_days": 90,
            "job_country_code_or": ["CA"],
            "job_title_or": [role],
            "limit": 25,
            "page": page,
        }
        response = requests.post(url, headers=headers, json=body, timeout=60)

        if response.status_code != 200:
            print(role, "page", page, "| status", response.status_code, "|", response.text)
            break

        data = response.json()
        with open(filename, "w") as f:
            json.dump(data, f, indent=2)
        print(datetime.now(), "| TheirStack |", role, "| page", page, "| status 200 |", len(data["data"]), "postings saved")
        time.sleep(1)

2026-11-22 19:49:14.065573 | TheirStack | data analyst | page 0 | status 200 | 25 postings saved
2026-11-22 19:49:20.403648 | TheirStack | data analyst | page 1 | status 200 | 25 postings saved
2026-11-22 19:49:24.131526 | TheirStack | data engineer | page 0 | status 200 | 25 postings saved
2026-11-22 19:49:30.267602 | TheirStack | data engineer | page 1 | status 200 | 25 postings saved
2026-11-22 19:49:35.774251 | TheirStack | data scientist | page 0 | status 200 | 25 postings saved
2026-11-22 19:49:42.781908 | TheirStack | data scientist | page 1 | status 200 | 25 postings saved


Combine findings into one table and check it 

In [19]:
rows = []
for name in sorted(os.listdir("../data/raw/theirstack")):
    if name.endswith(".json"):
        with open(f"../data/raw/theirstack/{name}") as f:
            saved = json.load(f)
        print(name, "|", len(saved["data"]), "postings")
        for job in saved["data"]:
            job["source_file"] = name
            rows.append(job)

theirstack = pd.DataFrame(rows)
print(len(theirstack), "rows |", theirstack["id"].nunique(), "unique postings")
print(theirstack["date_posted"].min(), "to", theirstack["date_posted"].max())
theirstack[["id", "job_title", "company", "location", "date_posted", "source_file"]].head()

theirstack_ca_data_analyst_p0.json | 25 postings
theirstack_ca_data_analyst_p1.json | 25 postings
theirstack_ca_data_analyst_test.json | 5 postings
theirstack_ca_data_engineer_p0.json | 25 postings
theirstack_ca_data_engineer_p1.json | 25 postings
theirstack_ca_data_scientist_p0.json | 25 postings
theirstack_ca_data_scientist_p1.json | 25 postings
155 rows | 149 unique postings
2026-11-17 to 2026-11-22


,id,job_title,company,location,date_posted,source_file
0,966358831,"Product Data Analyst, eCommerce [AQ-27829]",Aquent,"Toronto, Ontario",2026-11-22,theirstack_ca_data_analyst_p0.json
1,965542882,Data Analyst (Revenue & Margin),Trulioo,"Vancouver, British Columbia, Canada {{REMOTE}}",2026-11-22,theirstack_ca_data_analyst_p0.json
2,965542881,Senior Data Analyst (Revenue & Margin),Trulioo,"Vancouver, British Columbia, Canada {{REMOTE}}",2026-11-22,theirstack_ca_data_analyst_p0.json
3,967529933,"Business Analyst Co-op, Data Governance",Teck Resources Limited,"Vancouver, British Columbia, CA",2026-11-22,theirstack_ca_data_analyst_p0.json
4,962244502,"Analyst, Data & Business Intelligence (Temporary)",Bombardier,"Dorval, QC H4S 2A3",2026-11-20,theirstack_ca_data_analyst_p0.json


Keep the useful columns and drop the duplicates and save to csv file

In [20]:
keep = ["id", "job_title", "normalized_title", "company", "location", "state_code",
        "latitude", "longitude", "date_posted", "seniority", "remote", "hybrid",
        "min_annual_salary", "max_annual_salary", "salary_currency", "salary_string",
        "technology_slugs", "description", "url", "source_file"]

postings = theirstack[keep].drop_duplicates(subset="id")
postings["search_role"] = postings["source_file"].str.extract(r"theirstack_ca_(.+?)_(?:p\d|test)")[0].str.replace("_", " ")

postings.to_csv("../data/clean/theirstack_postings.csv", index=False)
print(postings.shape)
print(postings["search_role"].value_counts())

(149, 21)
search_role
data analyst      50
data scientist    50
data engineer     49
Name: count, dtype: int64


---
# 2. Web scraping: Job Bank wages by occupation

Request one page from Job bank and saving it

In [6]:
from bs4 import BeautifulSoup

url = "https://www.jobbank.gc.ca/wagereport/occupation/227147"   # data scientist, NOC 21211
headers = {"User-Agent": "DATA3463-FinalProject/1.0"}

response = requests.get(url, headers=headers, timeout=15)
response.raise_for_status()
html = response.text

os.makedirs("../data/raw/wages_html", exist_ok=True)
with open("../data/raw/wages_html/wages_data_scientist_noc21211.html", "w", encoding="utf-8") as f:
    f.write(html)

print(response.status_code, "|", len(html), "characters saved")

200 | 240840 characters saved


Parse the page and extract one row

In [7]:
soup = BeautifulSoup(html, "html.parser")
region_rows = soup.select("table#wage-occ-report tr.wage-region")
print(len(region_rows), "region rows")

first = region_rows[0]
province = first.select_one("td").get_text(strip=True)
region   = first.select_one("th").get_text(strip=True)
low      = float(first.select_one("td[headers$='header_min']")["data-wage"])
median   = float(first.select_one("td[headers$='header_avg']")["data-wage"])
high     = float(first.select_one("td[headers$='header_max']")["data-wage"])
print(province, "|", region, "|", low, median, high)

72 region rows
NL | Avalon Peninsula Region | 39.58 45.87 66.67


Loop into a list of dictionaries

In [10]:
wage_pages = [
    {"role": "data analyst",   "noc": "21223", "url": "https://www.jobbank.gc.ca/wagereport/occupation/17880"},
    {"role": "data engineer",  "noc": "21211", "url": "https://www.jobbank.gc.ca/wagereport/occupation/24510"},
    {"role": "data scientist", "noc": "21211", "url": "https://www.jobbank.gc.ca/wagereport/occupation/227147"},
]

def get_wage(row, column):
    cell = row.select_one(f"td[headers$='{column}']")
    if cell is None or not cell.get("data-wage"):
        return None                      # some regions have no wage published
    return float(cell["data-wage"])

rows = []
for page in wage_pages:
    filename = f"../data/raw/wages_html/wages_{page['role'].replace(' ', '_')}_noc{page['noc']}.html"

    if os.path.exists(filename):         # already saved, so read it instead of requesting again
        with open(filename, encoding="utf-8") as f:
            html = f.read()
    else:
        response = requests.get(page["url"], headers=headers, timeout=15)
        response.raise_for_status()
        html = response.text
        with open(filename, "w", encoding="utf-8") as f:
            f.write(html)
        print(datetime.now(), "|", page["role"], "| status", response.status_code, "| saved")
        time.sleep(5)                    # robots.txt Crawl-delay: 5

    soup = BeautifulSoup(html, "html.parser")
    for row in soup.select("table#wage-occ-report tr.wage-region, table#wage-occ-report tr.wage-province"):
        rows.append({
            "level": "province" if "wage-province" in row["class"] else "region",
            "role": page["role"],
            "noc": page["noc"],
            "province": row.select_one("td").get_text(strip=True),
            "region": row.select_one("th").get_text(strip=True),
            "wage_low": get_wage(row, "header_min"),
            "wage_median": get_wage(row, "header_avg"),
            "wage_high": get_wage(row, "header_max"),
        })

print("Rows:", len(rows), "First:", rows[0])

Rows: 255 First: {'level': 'province', 'role': 'data analyst', 'noc': '21223', 'province': 'NL', 'region': 'Newfoundland and Labrador', 'wage_low': 23.39, 'wage_median': 43.75, 'wage_high': 56.92}


Export to CSV

In [12]:
wages = pd.DataFrame(rows)
wages.to_csv("../data/clean/jobbank_wages_by_region.csv", index=False)

print(wages.shape)
print(wages.isna().sum())
wages.head()

(255, 8)
level            0
role             0
noc              0
province         0
region           0
wage_low       136
wage_median    136
wage_high      136
dtype: int64


,level,role,noc,province,region,wage_low,wage_median,wage_high
0,province,data analyst,21223,NL,Newfoundland and Labrador,23.39,43.75,56.92
1,region,data analyst,21223,NL,Avalon Peninsula Region,25.64,43.75,52.88
2,region,data analyst,21223,NL,Notre Dame-Central-Bonavista Bay Region,NaN,NaN,NaN
3,region,data analyst,21223,NL,South Coast–Burin Peninsula Region,NaN,NaN,NaN
4,region,data analyst,21223,NL,West Coast–Northern Peninsula–Labrador Region,NaN,NaN,NaN


---
# 3. Database / SQL: Job Bank open data

---
# 4. Technique beyond class (to be decided)